# SAM3 pipeline debug notebook (steps 1 to 3)

This notebook runs:
1. Pipeline step 1: dataset inference + concept map
2. Pipeline step 2: crops JSON creation with SAM3
3. Pipeline step 3: feature generation

Then it runs a focused `crops_to_json.py` demo for one tag and one image directory so you can inspect segmentation patch creation.

In [ ]:
from pathlib import Path
import sys

# --- Direct SAM3 input ---
ROOT_DIR = Path('/mnt/abka03/Projects/xl-vlms')
INPUT_DIR = Path('/mnt/abka03/xlvlm_data/Norwegen2024')
OUTPUT_DIR = ROOT_DIR / 'outputs' / 'sam3_debug'

# Set the images to segment and the tag prompt to use
IMAGE_PATHS = [
    Path('/mnt/abka03/xlvlm_data/Norwegen2024/20240422_094445.jpg'),
    Path('/mnt/abka03/xlvlm_data/Norwegen2024/20240419_192751.jpg'),
    Path('/mnt/abka03/xlvlm_data/Norwegen2024/20240422_075824.jpg'),
]
TAG_NAME = 'blue'
IMAGE_SIZE_WIDTH = 512
# SAM3 knobs
DEVICE = 'cuda:0'
SEGMENTATION_CONFIDENCE = -1  # use adaptive thresholding in sam3_utils
DETECTION_BATCH_SIZE = 2
CONCEPT_MASKS_PER_IMAGE = 5   # minimum keep for adaptive mode

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT_DIR  :', ROOT_DIR)
print('INPUT_DIR :', INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('IMAGE_PATHS:')
for p in IMAGE_PATHS:
    print('  -', p)
print('TAG_NAME  :', TAG_NAME)

In [ ]:
# Direct SAM3 overlay demo using sam3_utils only.
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

_ensure_repo_root_on_sys_path()
from preprocessing.crops_to_json import _load_and_resize
from src.sam3_utils import load_sam3, predict_bboxes_and_masks_for_tag_sam3_batched

image_paths = []
for image_path in IMAGE_PATHS:
    if not image_path.is_file():
        raise FileNotFoundError(f'Image not found: {image_path}')
    image_paths.append(image_path)

pil_images = []
for image_path in image_paths:
    print(f'Loading and resizing {image_path.name!r}...')
    img, _, _ = _load_and_resize(str(image_path), IMAGE_SIZE_WIDTH)
    pil_images.append(img)

print('Loading SAM3 model...')
model = load_sam3(device=DEVICE, confidence_threshold=SEGMENTATION_CONFIDENCE)
model['minimum_keep'] = CONCEPT_MASKS_PER_IMAGE
print('SAM3 model loaded')
print('  confidence_threshold =', model.get('confidence_threshold'))
print('  minimum_keep         =', model.get('minimum_keep'))

pairs_per_img = predict_bboxes_and_masks_for_tag_sam3_batched(
    model,
    pil_images,
    TAG_NAME,
    batch_size=DETECTION_BATCH_SIZE,
    topn=CONCEPT_MASKS_PER_IMAGE,
)

for image_path, img, pairs in zip(image_paths, pil_images, pairs_per_img):
    img_np = np.array(img)
    union_mask = None

    for bbox, mask in pairs:
        if mask is None:
            continue
        mask_bool = np.asarray(mask, dtype=bool)
        if mask_bool.shape[:2] != img_np.shape[:2]:
            mask_bool = np.array(
                Image.fromarray((mask_bool > 0).astype(np.uint8) * 255).resize(
                    (img_np.shape[1], img_np.shape[0]), Image.NEAREST
                )
            ) > 127
        union_mask = mask_bool if union_mask is None else (union_mask | mask_bool)

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img_np)
    if union_mask is not None:
        ax.imshow(np.where(union_mask[..., None], np.array([0, 255, 0], dtype=np.uint8), 0), alpha=0.35)
    ax.set_title(f'{image_path.name} — {TAG_NAME} mask overlay')
    ax.axis('off')
    plt.show()
    print(f'[{image_path.name}] masks kept: {len(pairs)}')